In [ ]:
# Feature Engineering — Encoding & Scaling

In [ ]:
# Categorical Encoding

import numpy as np
import pandas as pd

# Synthetic house price dataset (mirrors Week 6 House Price project, adds categoricals
# that the original numeric-only dataset didn't have, for encoding practice)
rng = np.random.default_rng(42)
n = 500

neighborhood = rng.choice(['Downtown', 'Suburb', 'Rural'], size=n, p=[0.4, 0.4, 0.2])
house_style = rng.choice(['1Story', '2Story', 'Split'], size=n)
sqft = rng.normal(1800, 500, n).clip(500, 4000)
bedrooms = rng.integers(1, 6, n)
age = rng.integers(0, 80, n)
distance_to_city = rng.normal(15, 8, n).clip(0.5, 60)

neighborhood_premium = {'Downtown': 1.4, 'Suburb': 1.1, 'Rural': 0.8}
style_premium = {'1Story': 1.0, '2Story': 1.15, 'Split': 1.05}

base_price = (
    sqft * 120
    + bedrooms * 8000
    - age * 500
    - distance_to_city * 900
)
price = base_price * np.array([neighborhood_premium[v] for v in neighborhood]) \
              * np.array([style_premium[v] for v in house_style])
price = price + rng.normal(0, 15000, n)  # noise

df = pd.DataFrame({
    'neighborhood': neighborhood,
    'house_style': house_style,
    'sqft': sqft,
    'bedrooms': bedrooms,
    'age': age,
    'distance_to_city': distance_to_city,
    'price': price
})

df.head()


In [ ]:
df.info()
print()
print(df[['neighborhood', 'house_style']].nunique())


In [ ]:
# Train/Test Split BEFORE Any Transform

from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = ['neighborhood', 'house_style']
numeric_cols = ['sqft', 'bedrooms', 'age', 'distance_to_city']

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


In [ ]:
# Scaling

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', Ridge(alpha=1.0))
])

pipeline


In [ ]:
# Fit Pipeline — Fit Happens on Train Only

pipeline.fit(X_train, y_train)

from sklearn.metrics import mean_squared_error
preds = pipeline.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
print(f"Test RMSE (pipeline, encoded+scaled): {rmse:.2f}")

# Inspect what the encoder learned (categories) and scaler learned (mean/std)
fitted_ohe = pipeline.named_steps['preprocess'].named_transformers_['cat']
fitted_scaler = pipeline.named_steps['preprocess'].named_transformers_['num']

print("\nOneHotEncoder categories learned from TRAIN only:")
for col, cats in zip(categorical_cols, fitted_ohe.categories_):
    print(f"  {col}: {list(cats)}")

print("\nStandardScaler mean (train only):", np.round(fitted_scaler.mean_, 2))
print("StandardScaler std (train only):", np.round(fitted_scaler.scale_, 2))


In [ ]:
# Leakage Check — Prove Test Data Was Never Used

manual_train_mean = X_train[numeric_cols].mean().values
manual_full_mean = X[numeric_cols].mean().values

print("Scaler's stored mean:      ", np.round(fitted_scaler.mean_, 2))
print("Manually computed TRAIN mean:", np.round(manual_train_mean, 2))
print("Manually computed FULL mean: ", np.round(manual_full_mean, 2))

match_train = np.allclose(fitted_scaler.mean_, manual_train_mean)
match_full = np.allclose(fitted_scaler.mean_, manual_full_mean)
print(f"\nScaler matches TRAIN-only mean: {match_train}")
print(f"Scaler matches FULL-dataset mean: {match_full}  (should be False unless coincidence)")


In [ ]:
# Cross-Validating the Full Pipeline (Leak-Free)

from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = -cross_val_score(pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')

print("RMSE per fold:", np.round(cv_scores, 2))
print(f"CV Mean RMSE: {cv_scores.mean():.2f}  Std: {cv_scores.std():.2f}")
